# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [85]:
import asyncio
import json
import re
import os
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI
from dotenv import load_dotenv

# To load Environment variables
load_dotenv()

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [88]:
# Loading the Job Snippets and Golden dataset in dictionary
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [89]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    user_prompt = f"""Read the job snippet and extract following fields from the job description: company, role, years_experience_required. Output the result in raw JSON format."
    Job snippet: "{snippet_text}"

    Provide your final response as a raw JSON object only
    """
    return [{"role": "user", "content": user_prompt}]
            


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    user_prompt = f"""Read the job snippet and extract following fields from the job description: company, role, years_experience_required. Output the result in raw JSON format.
    
    Examples:
    Job snippet: Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.
    Output: {{"company": "Acme Corp"", "role": "Senior Software Engineer", "years_experience_required": 5}}

    Job snippet: Hooli is looking for a Junior Frontend Developer. Fresh grads welcome — no prior experience required. We care about curiosity and willingness to learn. JavaScript, React, and CSS fundamentals expected.
    Output: {{"company": "Hooli", "role": "Junior Frontend Developer", "years_experience_required": 0}}

    Job snippet: Cyberdyne Systems is hiring. Role: AI/ML Research Scientist. PhD preferred but not required. Strong publication record in deep learning or reinforcement learning. We don't list a specific years requirement — we hire on demonstrated impact.
    Output: {{"company": "Cyberdyne Systems"", "role": "AI/ML Research Scientist", "years_experience_required": null}}

    Job snippet: "{snippet_text}"

    Provide your final response as a raw JSON object only
    """

    return [{"role": "user", "content": user_prompt}]



def prompt_structured(snippet_text: str) -> list[dict]:
    system_prompt = """You are an expert data extraction assistant and specialized in job description analysis. Read the Job Snippet mentioned below and perform following tasks:
    1. Extract following fields from the Job Snippet: company, role, years_experience_required.
    2. Do  not assume if any information is missing and return null for that field.
    3. years_experience_required should be number
    4. Output the result in raw JSON format with keys as shown in sample below:
    Output format sample - {"company": <Name of hiring company> , "role": <Job role offered in Job Snippet>, "years_experience_required": <Minimum years of experience required>} 
    
    Examples:
    Job snippet: Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.
    Output: {"company": "Acme Corp"", "role": "Senior Software Engineer", "years_experience_required": 5}

    Job snippet: Hooli is looking for a Junior Frontend Developer. Fresh grads welcome — no prior experience required. We care about curiosity and willingness to learn. JavaScript, React, and CSS fundamentals expected.
    Output: {"company": "Hooli", "role": "Junior Frontend Developer", "years_experience_required": 0}

    Job snippet: Cyberdyne Systems is hiring. Role: AI/ML Research Scientist. PhD preferred but not required. Strong publication record in deep learning or reinforcement learning. We don't list a specific years requirement — we hire on demonstrated impact.
    Output: {"company": "Cyberdyne Systems"", "role": "AI/ML Research Scientist", "years_experience_required": null}

    Provide your final response as a raw JSON object only.

    """
    user_prompt = f"Job snippet: {snippet_text}"

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
        ]

def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    user_prompt = f"""You are an expert data extraction assistant. Your task is to extract the company, role, years_experience_required from the provided job snippet and output the result in a clean, raw JSON format.

    Perform the task by step-by-step thinking process to ensure correct result:
    1. Read the job snippet thoroughly
    2. Identify the company name mentioned in the snippet
    3. Identify the role or job title mentioned in the snippet
    4. Identify the minimum years of experience required for the role. If not mentioned, return null for that field. years_experience_required should be a number
    5. Output the result in raw JSON format with exactly these keys - 
        - "company": <Name of hiring company>
        - "role": <Job role offered in Job Snippet>
        - "years_experience_required": <Minimum years of experience required or null in case value is missing>
        - "reasoning": <description of step by step thinking process>

    Job Snippet: {snippet_text}

    Provide your final response as a raw JSON object only.
    """

    return [{"role": "user", "content": user_prompt}]

    
# Creating a dictionairy of all prompt strategy functions
STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [90]:
def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    #If raw LLM response is not available, retun None
    if not text:
        return None

    # Remove overall leading/trailing whitespace
    cleaned_text = text.strip()

    #Regex to capture content inside markdown code blocks and handle ```json ... ``` and multiline content
    markdown_pattern = r"^```(?:json)?\s*(.*?)\s*```$"
    match = re.search(markdown_pattern, cleaned_text, re.DOTALL | re.IGNORECASE)
    if match:
        cleaned_text = match.group(1).strip()

    # Decoding structured JSON response
    try:
        return json.loads(cleaned_text)
    except json.JSONDecodeError:
        return None

    


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    start_time = time.perf_counter()

    response = await client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        messages=STRATEGIES[strategy_name](snippet["snippet"])
    )
    
    latency_sec = time.perf_counter() - start_time
    raw_response = response.choices[0].message.content
    cost_usd = response.usage.prompt_tokens * RATES[MODEL]["in"] + response.usage.completion_tokens * RATES[MODEL]["out"] 
   
    return {
        "strategy_name": strategy_name,
        "snippet_ID": snippet["id"],
        "raw_response": raw_response,
        "parsed_response": parse_response(raw_response),
        "cost_usd": cost_usd,
        "latency_sec": latency_sec
            
    }
    


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    tasks = [run_one(strategy_name, snippet) for snippet in snippets for strategy_name in STRATEGIES]
    return await asyncio.gather(*tasks)

In [91]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy_name': 'zero_shot',
 'snippet_ID': 'j01',
 'raw_response': '```json\n{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": "5+"\n}\n```',
 'parsed_response': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': '5+'},
 'cost_usd': 3.465e-05,
 'latency_sec': 1.9930399580625817}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [92]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    score = 0
    if extracted is None:
        return score

    for key in ["company", "role", "years_experience_required"]:
        if str(extracted[key]).strip().lower() == str(gold[key]).strip().lower():
            score +=1

    return score 




async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    
    """Use gpt-4o as a judge. Return integer 1-4.
    
    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    judge_prompt = f""" You are a strict evaluator of LLM responses. Evalue Model Response against Gold Answer and give a score from 1 to 4 as per criteria below. Job Snippet is also added from which model response is generated:

    Score from 1 to 4:
    4 — all three fields correct
    3 — two of three correct, no fabricated data
    2 — one of three correct, or fabricated a field
    1 — none correct or unparsable   

    Job Snippet: {snippet_text}
    Mode Response: {extracted}
    Gold Answer: {gold}

    Return only score in integer format

    """

    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=TEMPERATURE,
        messages=[{"role":"user", "content": judge_prompt}]
    )

    return int(response.choices[0].message.content.strip())

In [93]:
# Apply scoring to all 40 results
# TODO: loop through results, attach accuracy + parse_success + llm_judge_score to each row
scored = []   # list of result dicts with scoring fields added

for result in results:
    gold_record = golden[result["snippet_ID"]]
    snippet_text = next(s["snippet"] for s in snippets if s["id"] == result["snippet_ID"])
    model_response = result["parsed_response"]

    result["accuracy"] = score_accuracy(model_response,gold_record)
    result["parse_success"] = result["parsed_response"] is not None
    result["llm_judge_score"] = await score_llm_judge(
        snippet_text, 
        model_response, 
        gold_record
    )
    scored.append(result)
    

print(f'Scored {len(scored)} results.')

Scored 40 results.


## Step 5 — Build the comparison table

In [111]:
# Summarizing the records by Prompt Strategy
df = pd.DataFrame(scored)

summary = df.groupby('strategy_name').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_sec': 'median',
}).round(3)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy_name,,,,,
cot,2.8,1.0,3.5,0.001,2.833
few_shot,3.0,1.0,3.8,0.001,2.186
structured,3.0,1.0,3.8,0.001,2.144
zero_shot,2.3,1.0,3.6,0.000,2.175


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```